In [1]:
%load_ext autoreload
%autoreload 2
%autosave 30

Autosaving every 30 seconds


# Evaluate the raw ensemble and develop a first baseline

In this notebook the train, valid and test datasets are scored using the crps, energy score and variogram score.\
This will be done per lead time.

In [2]:
import dask as da  # noqa: F401
import numpy as np
import pandas as pd
import torch
import xarray as xr
from tqdm import tqdm

import wandb
from genpp.data.weatherbench2 import (
    BASE_DIR,
    FC_VARS,
    FORECAST_ENS_FLAT_AGG_PATH,
    FORECAST_ENS_PATH,
    OBSERVATIONS_FLAT_PATH,
    TEST_PREDICTIONS,
    TRAIN_PREDICTIONS,
    VAL_PREDICTIONS,
)
from genpp.eval.utils import compute_scores_per_leadtime, save_scores_df
from genpp.models.scores import (
    EnergyScore,
    EnsembleCRPS,
    MultiScaleEnergyScore,
    MultiScalePatchwiseEnergyScore,
    PatchwiseEnergyScore,
    VariogramScore,
)

In [3]:
def load_data(predictions: pd.MultiIndex) -> tuple[xr.DataArray, xr.DataArray]:
    """Load data for evaluation.

    Args:
        predictions: MultiIndex with levels ('time', 'prediction_timedelta')
            specifying which (init_time, lead_time) pairs to evaluate.
    """
    if not isinstance(predictions, pd.MultiIndex):
        raise TypeError("predictions must be a pandas.MultiIndex")

    expected_names = ["time", "prediction_timedelta"]
    if list(predictions.names) != expected_names:
        predictions = predictions.set_names(expected_names)

    ens = xr.open_dataset(
        FORECAST_ENS_PATH,
        chunks={
            "time": "auto",
            "number": -1,
            "latitude": -1,
            "longitude": -1,
            "level": -1,
        },
    )
    ens = ens[FC_VARS]

    # This already contains the cleaned time data
    ens_flat = xr.open_dataset(FORECAST_ENS_FLAT_AGG_PATH)

    ens = (
        ens.sel(time=ens_flat.time)
        .to_dataarray("feature")
        .stack(prediction=("time", "prediction_timedelta"))
        .transpose("prediction", "number", "feature", "longitude", "latitude")
    )

    ens_prediction_index = ens.indexes["prediction"]
    missing = predictions.difference(ens_prediction_index)
    if len(missing) > 0:
        raise KeyError(f"{len(missing)} requested predictions are not available in ensemble data")

    # Slice to exactly the split-specific MultiIndex
    ens = ens.sel(prediction=predictions)

    # Compute target observation times from selected (time, prediction_timedelta)
    prediction_times = ens.time.values + ens.prediction_timedelta.values

    # Load corresponding observations
    obs = (
        xr.open_dataset(OBSERVATIONS_FLAT_PATH)
        .to_dataarray("feature")
        .sel(feature=FC_VARS)  # get them in the correct order
        .sel(time=prediction_times)
        .rename({"time": "prediction_time"})
        .transpose("prediction_time", "feature", "longitude", "latitude")
    )

    # Assign the selected prediction MultiIndex and align by it
    selected_predictions = ens.indexes["prediction"]
    obs = obs.assign_coords(prediction=("prediction_time", selected_predictions)).swap_dims(
        {"prediction_time": "prediction"}
    )

    assert np.all(prediction_times == obs.prediction_time.values)

    obs = obs.drop_vars("prediction_time")
    return ens, obs

## Compute the Scores

In [4]:
def score(ens: xr.DataArray, obs: xr.DataArray, skip_variogram: bool):
    """Compute scores given ensemble and observations.
    Args:
        ens (xr.DataArray): Ensemble predictions with dimensions (time, number, feature, longitude, latitude)
        obs (xr.DataArray): Observation data with dimensions (time, feature, longitude, latitude)
        skip_variogram (bool): Whether to skip variogram score computation, due to long computation times

    """
    requested_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    compute_device = requested_device
    print(f"Using device: {compute_device}")

    # Keep full tensors on CPU and move one sample at a time to avoid GPU OOM
    ens_tensor = torch.from_numpy(ens.to_numpy())
    obs_tensor = torch.from_numpy(obs.to_numpy())

    # Compute the scores
    crps_ens = EnsembleCRPS().to(compute_device)
    es = EnergyScore(clamp=False).to(compute_device)
    pw_es = PatchwiseEnergyScore(
        beta=1.0, clamp=True, patch_size=3, normalize=True, unbiased=True
    ).to(compute_device)
    ms_es = MultiScaleEnergyScore(
        beta=1.0,
        clamp=True,
        blur_kernel_sizes=[3, 7],
        scale_weights=[1.0, 1.0, 1.0],
        normalize=True,
        unbiased=True,
    ).to(compute_device)
    mspw_es = MultiScalePatchwiseEnergyScore(
        beta=1.0,
        clamp=True,
        blur_kernel_sizes=[3, 7],
        scale_weights=[1.0, 1.0, 1.0],
        patch_size=3,
        normalize=True,
        unbiased=True,
    ).to(compute_device)
    vs = VariogramScore(p=0.5).to(compute_device)

    crps_per_margin_list = []
    energy_score_per_var_u_list = []
    energy_score_full_u_list = []
    pw_es_pv_list, pw_es_full_list = [], []
    ms_es_pv_list, ms_es_full_list = [], []
    mspw_es_pv_list, mspw_es_full_list = [], []

    n_samples = ens_tensor.shape[0]
    switched_to_cpu = False

    for i in tqdm(range(n_samples), total=n_samples):
        ens_i = ens_tensor[i : i + 1]
        obs_i = obs_tensor[i : i + 1]

        try:
            ens_i_d = ens_i.to(compute_device)
            obs_i_d = obs_i.to(compute_device)
            crps_out = crps_ens(ens_i_d, obs_i_d)
            es_per_var_out = es(ens_i_d, obs_i_d, mode="per_var")
            es_full_out = es(ens_i_d, obs_i_d, mode="complete")
            pw_es_pv_out = pw_es(ens_i_d, obs_i_d, mode="per_var")
            pw_es_full_out = pw_es(ens_i_d, obs_i_d, mode="complete")
            ms_es_pv_out = ms_es(ens_i_d, obs_i_d, mode="per_var")
            ms_es_full_out = ms_es(ens_i_d, obs_i_d, mode="complete")
            mspw_es_pv_out = mspw_es(ens_i_d, obs_i_d, mode="per_var")
            mspw_es_full_out = mspw_es(ens_i_d, obs_i_d, mode="complete")
        except torch.OutOfMemoryError:
            if compute_device.type != "cuda":
                raise
            torch.cuda.empty_cache()
            compute_device = torch.device("cpu")
            switched_to_cpu = True
            crps_ens = crps_ens.to(compute_device)
            es = es.to(compute_device)
            pw_es = pw_es.to(compute_device)
            ms_es = ms_es.to(compute_device)
            mspw_es = mspw_es.to(compute_device)
            vs = vs.to(compute_device)
            ens_i_d = ens_i.to(compute_device)
            obs_i_d = obs_i.to(compute_device)
            crps_out = crps_ens(ens_i_d, obs_i_d)
            es_per_var_out = es(ens_i_d, obs_i_d, mode="per_var")
            es_full_out = es(ens_i_d, obs_i_d, mode="complete")
            pw_es_pv_out = pw_es(ens_i_d, obs_i_d, mode="per_var")
            pw_es_full_out = pw_es(ens_i_d, obs_i_d, mode="complete")
            ms_es_pv_out = ms_es(ens_i_d, obs_i_d, mode="per_var")
            ms_es_full_out = ms_es(ens_i_d, obs_i_d, mode="complete")
            mspw_es_pv_out = mspw_es(ens_i_d, obs_i_d, mode="per_var")
            mspw_es_full_out = mspw_es(ens_i_d, obs_i_d, mode="complete")

        crps_per_margin_list.append(crps_out.cpu())
        energy_score_per_var_u_list.append(es_per_var_out.cpu())
        energy_score_full_u_list.append(es_full_out.cpu())
        pw_es_pv_list.append(pw_es_pv_out.cpu())
        pw_es_full_list.append(pw_es_full_out.cpu())
        ms_es_pv_list.append(ms_es_pv_out.cpu())
        ms_es_full_list.append(ms_es_full_out.cpu())
        mspw_es_pv_list.append(mspw_es_pv_out.cpu())
        mspw_es_full_list.append(mspw_es_full_out.cpu())

    if switched_to_cpu and requested_device.type == "cuda":
        print("CUDA OOM encountered; switched score computation to CPU.")

    if not skip_variogram:
        vss_per_var_list = []
        vss_full_u_list = []

        for i in tqdm(range(n_samples), total=n_samples):
            ens_i = ens_tensor[i : i + 1].to(compute_device)
            obs_i = obs_tensor[i : i + 1].to(compute_device)
            vss_per_var_list.append(vs(ens_i, obs_i, mode="per_var").cpu())
            vss_full_u_list.append(vs(ens_i, obs_i, mode="complete").cpu())

        variogram_score_per_var_u = torch.cat(vss_per_var_list, dim=0)
        variogram_score_full_u = torch.cat(vss_full_u_list, dim=0)

    crps_per_margin = torch.cat(crps_per_margin_list, dim=0)
    energy_score_per_var_u = torch.cat(energy_score_per_var_u_list, dim=0)
    energy_score_full_u = torch.cat(energy_score_full_u_list, dim=0)
    pw_es_per_var_u = torch.cat(pw_es_pv_list, dim=0)
    pw_es_full_u = torch.cat(pw_es_full_list, dim=0)
    ms_es_per_var_u = torch.cat(ms_es_pv_list, dim=0)
    ms_es_full_u = torch.cat(ms_es_full_list, dim=0)
    mspw_es_per_var_u = torch.cat(mspw_es_pv_list, dim=0)
    mspw_es_full_u = torch.cat(mspw_es_full_list, dim=0)

    scores = compute_scores_per_leadtime(
        prediction_timedeltas=ens.prediction_timedelta.values,
        crpss=crps_per_margin,
        ess_per_var=energy_score_per_var_u,
        ess_complete=energy_score_full_u,
        vss_per_var=variogram_score_per_var_u if not skip_variogram else None,  # type: ignore
        vss_complete=variogram_score_full_u if not skip_variogram else None,  # type: ignore
        pw_es_per_var=pw_es_per_var_u,
        pw_es_complete=pw_es_full_u,
        ms_es_per_var=ms_es_per_var_u,
        ms_es_complete=ms_es_full_u,
        mspw_es_per_var=mspw_es_per_var_u,
        mspw_es_complete=mspw_es_full_u,
        method=None,
    )
    return scores

## Put everything together

In [5]:
split_predictions = {
    "train": TRAIN_PREDICTIONS,
    "val": VAL_PREDICTIONS,
    "test": TEST_PREDICTIONS,
}

scores = {}
for split, predictions in split_predictions.items():
    if not isinstance(predictions, pd.MultiIndex):
        raise TypeError(f"{split} predictions must be a pandas.MultiIndex")

    expected_names = ["time", "prediction_timedelta"]
    if list(predictions.names) != expected_names:
        predictions = predictions.set_names(expected_names)

    print(
        f"Scoring split: {split} ({len(predictions)} predictions) "
        f"with index names={predictions.names}"
    )
    ens, obs = load_data(predictions)
    scores[split] = score(ens, obs, skip_variogram=False)

Scoring split: train (10905 predictions) with index names=['time', 'prediction_timedelta']
Using device: cuda


100%|██████████| 10905/10905 [04:04<00:00, 44.60it/s]


Processing leadtime 24h with 2185 samples
Processing leadtime 48h with 2183 samples
Processing leadtime 72h with 2181 samples
Processing leadtime 96h with 2179 samples
Processing leadtime 120h with 2177 samples
Scoring split: val (3620 predictions) with index names=['time', 'prediction_timedelta']
Using device: cuda


100%|██████████| 3620/3620 [01:25<00:00, 42.26it/s]


Processing leadtime 24h with 728 samples
Processing leadtime 48h with 726 samples
Processing leadtime 72h with 724 samples
Processing leadtime 96h with 722 samples
Processing leadtime 120h with 720 samples
Scoring split: test (3620 predictions) with index names=['time', 'prediction_timedelta']
Using device: cuda


100%|██████████| 3620/3620 [01:08<00:00, 52.86it/s]


Processing leadtime 24h with 728 samples
Processing leadtime 48h with 726 samples
Processing leadtime 72h with 724 samples
Processing leadtime 96h with 722 samples
Processing leadtime 120h with 720 samples


## Create a dummy wandb run and save the scores

In [ ]:
run = wandb.init(
    project="genpp",
    name="ifs-raw-ensemble-baseline-scores",
    tags=["baseline", "best", "final"],
    config={
        "method": "raw_ensemble",
        "description": "Baseline scores for raw IFS ensemble forecasts",
    },
    dir=BASE_DIR.parent.parent / "outputs" / "BASELINE",
)

wandb: Currently logged in as: feik to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [6]:
# ID of the baseline run
run_id = "feik/genpp/utgo5npv"

# Create a wandb run to save the scores
api = wandb.Api()
run = api.run(run_id)
short_run_id = run_id.split("/")[-1]
try:
    # If the run already exists, resume it
    with wandb.init(
        entity=run.entity, project=run.project, id=short_run_id, resume="must"
    ) as active_run:
        active_run.summary.update(scores)

except Exception:
    print(f"Run with ID {run_id} not found. Creating a new run.")
    # If it does not exist, create a new one
    run = wandb.init(
        project="genpp",
        name="ifs-raw-ensemble-baseline-scores",
        tags=["baseline", "best", "final"],
        config={
            "name": "raw_ensemble",
            "description": "Baseline scores for raw IFS ensemble forecasts",
        },
        dir=BASE_DIR.parent.parent / "outputs" / "BASELINE",
        resume="must",
    )

    run.summary.update(scores)

# Also log as a table
records = []
for dataset, metrics in scores.items():
    for metric_name, horizons in metrics.items():
        for horizon, value in horizons.items():
            records.append(("raw_ensemble", dataset, metric_name, horizon, value))
df = pd.DataFrame(records, columns=["method", "dataset", "metric", "horizon", "value"])

save_scores_df(df=df, run_path="/".join(run.path))

try:
    run.finish()
except AttributeError:
    pass

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /home/feik/.netrc.
wandb: Currently logged in as: feik to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


In [7]:
pd.set_option("display.max_rows", 200)
df

,method,dataset,metric,horizon,value
0,raw_ensemble,train,CRPS_combined,24h,0.485524
1,raw_ensemble,train,CRPS_combined,48h,0.553438
2,raw_ensemble,train,CRPS_combined,72h,0.623476
3,raw_ensemble,train,CRPS_combined,96h,0.721191
4,raw_ensemble,train,CRPS_combined,120h,0.839353
...,...,...,...,...,...
265,raw_ensemble,test,MultiScalePatchwiseEnergyScore_10m_windspeed,24h,0.371519
266,raw_ensemble,test,MultiScalePatchwiseEnergyScore_10m_windspeed,48h,0.427703
267,raw_ensemble,test,MultiScalePatchwiseEnergyScore_10m_windspeed,72h,0.492868
268,raw_ensemble,test,MultiScalePatchwiseEnergyScore_10m_windspeed,96h,0.582985
